In [2]:
!pip install -q -U langchain langchain-google-genai langchain-community pypdf faiss-cpu

In [18]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"]=userdata.get("GEMINI_API_KEYS")
MODEL="gemini-3.5-flash-lite"
EMB_MODEL="gemini-embedding-001"

In [11]:
#1Prepare Document
#1.1.Upload PDF
from google.colab import files
uploaded=files.upload()
pdf_name=list(uploaded.keys())[0]
print(f"PDF:{pdf_name} uploaded sucessfully")

Saving school_faq.pdf to school_faq (2).pdf
PDF:school_faq (2).pdf uploaded sucessfully


In [12]:
#1.2Extract text from pdf
from langchain_community.document_loaders import PyPDFLoader
docs=PyPDFLoader(pdf_name).load()
print("NO OF PAGES IN DOCUMENT: ",len(docs))
print(docs)

NO OF PAGES IN DOCUMENT:  2
[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-09-06T19:15:21+05:30', 'author': 'kushal yn', 'moddate': '2026-09-06T19:15:21+05:30', 'source': 'school_faq (2).pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content="Q: What are the school working hours?  \nA: The school working hours are from 8:00 AM to 4:00 PM.  \n \nQ: How can parents apply for leave for their child?  \nA: Parents can submit a leave request through the school's prescribed leave application process.  \n \nQ: How can parents check their child's attendance?  \nA: Parents can check attendance through the school's available attendance system.  \n \nQ: How can students check their examination results?  \nA: Students can check their examination results through the school's result system or by contacting the \nschool administration. \n \nQ: How can parents pay school fees?  \nA: Parents can pay school fees through the pa

In [14]:
#1.3Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=150)
chunks=splitter.split_documents(docs)
print(f"CHUNKING DONE WITH {len(chunks)} chunks")

CHUNKING DONE WITH 5 chunks


In [19]:
#1.4 Embedding
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
embeddings=GoogleGenerativeAIEmbeddings(model=EMB_MODEL)
vectorstore=FAISS.from_documents(chunks,embeddings)

In [21]:
#1.5 Retreival of text (object creation)
retriever=vectorstore.as_retriever(search_kwargs={"k":3})

In [22]:
llm=ChatGoogleGenerativeAI(model=MODEL)

In [35]:
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage,ToolMessage
@tool
def search_info(query: str) -> str:
  """
  Search the school knowledge base and provide a
    formatted answer to questions about school information,
    rules, admission, fees, facilities, timings, departments,
    and other information stored in school documents.
  """
  prompt=ChatPromptTemplate([
      ('system',"""
      When user asks about school info then search in knowledge base
      You are a school information assistant.

      Answer the user's question using ONLY the context provided below.

      If the answer is not present in the context, say:
      "I could not find this information in the school documents."
      if they ask for summary tell summary of overall content you have
      Give a clear and concise answer."""),
  ('human','CONTEXT:{context}-------QUESTION:{question}')

  ])
  def format_docs(docs):
    return ("\n--------------\n".join(d.page_content for d in docs))

  rag_chain=(
      {'context':retriever | format_docs, 'question':RunnablePassthrough()}
      |prompt
      |llm
      |StrOutputParser()
  )
  res=rag_chain.invoke(query)
  return res

@tool
def percentage_cal(obtained: int, total: int) -> float:
  "TOOL TO CALCULATE PERCENTAGE OF STUDENT"
  return (obtained/total)*100

@tool
def attandance_elgibility(present: int,total: int) -> bool:
  "TOOL TO CHECK ATTANDANCE ELIGIBILTY where minimum attandance is greater than or equal to 80%"
  result=(present/total)*100
  if result >=80:
    return True
  else:
    return False

llm_with_tools=llm.bind_tools([search_info,percentage_cal,attandance_elgibility])

def chat(query):
  messages=[]
  messages.append(HumanMessage(content=query))
  AI=llm_with_tools.invoke(messages)
  messages.append(AI)


  if not AI.tool_calls:
    return

  tool_call=AI.tool_calls[0]
  tools={
      "search_info":search_info,
      "percentage_cal":percentage_cal,
      "attandance_elgibility":attandance_elgibility
  }
  selected_tool=tools[tool_call["name"]]
  tool_result=selected_tool.invoke(tool_call["args"])

  tool_messages=ToolMessage(
      content=str(tool_result),
      name=tool_call["name"],
      tool_call_id=tool_call["id"]
  )
  messages.append(tool_messages)

  final_output=llm_with_tools.invoke(messages)
  return final_output.content[0]['text']

print("Enter bye/exit/quit to terminate session")
while True:

  user = input("Enter your query: ").strip()
  if user.lower() in ['exit', 'bye', 'quit']:
    print("exited")
    break
  output = chat(user)
  print(output)

Enter bye/exit/quit to terminate session
Enter your query: summarize about this schoo;
Here is a summary of the school based on the available information:

* **School Timings:** 8:00 AM to 4:00 PM.
* **Admissions:** Requires submitting specified documents during the admission process.
* **Fees:** Payments can be made using the school's provided payment methods.
* **Attendance & Leave:** Parents can check their child's attendance and apply for student leave through the school's systems.
* **Examinations & Results:** Students can view their results through the school's result system or by contacting the administration.
* **Communication:** Parents can connect with class teachers via the school's designated communication channels.
Enter your query: tell me communication for parents with this school
Parents can receive announcements and stay connected through the school's official communication channels. 

If you need specific details such as the primary communication app, contact numbers,